SVM (RBF): Phase-Grouped 5-Fold Cross-Validation with Validation-Only Threshold Calibration,
Equal Tuning Budget, AUPRC Reporting, Final Independent Test (Phases 14-16),
and Per-Image Error Maps

Workflow:
1) Keep the existing manual Train/Test split.
2) Assign eruption Phases 1-16 from timestamps.
3) Use fixed phase-grouped 5-fold CV only on Phases 1-13.
4) Evaluate an equal tuning budget of 30 RBF-SVM hyperparameter configurations (C and gamma).
5) For every CV fold, fit StandardScaler + SVM ONLY on that fold's training images.
6) Calibrate the SVM decision threshold ONLY on the validation fold using the precision-recall curve
   and select the threshold that maximizes validation F1/Dice.
7) Report AUPRC (Average Precision) together with threshold-dependent metrics.
8) Select the best hyperparameter configuration by mean calibrated validation F1/Dice,
   with mean AUPRC retained as a threshold-independent performance measure.
9) Re-run 5-fold CV with the selected hyperparameters using ALL validation pixels.
10) Define the final locked decision threshold as the median of the five validation-optimal thresholds.
11) Retrain the final SVM on all Phase 1-13 training images.
12) Evaluate the locked Phase 14-16 test set ONCE using the locked threshold.
13) Save per-image, phase-level, overall pooled metrics, AUPRC, predictions, and error maps.

IMPORTANT:
- SVM is pixel-based: each pixel uses the same 3 Ash-RGB channels as features.
- Training uses a reproducible class-balanced pixel sample for computational feasibility.
- Because the training sample is already balanced, class_weight is set to None.
- The threshold is calibrated against the natural class prevalence of held-out validation images.
- Validation/test pixels are never used to fit the StandardScaler or SVM.
- Final independent test data are never used for hyperparameter or threshold selection.


In [1]:
# ============================================================
# CELL 1 - IMPORT LIBRARIES
# ============================================================

from pathlib import Path
from collections import defaultdict
import gc
import json
import math
import random
import re
import time

import joblib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import rasterio

from PIL import Image

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print("NumPy version :", np.__version__)
print("Pandas version:", pd.__version__)


NumPy version : 2.5.2
Pandas version: 3.0.5


In [2]:
# ============================================================
# CELL 2 - MAIN CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path(r"D:\Experiment Ajarn")
IMAGE_DIR = BASE_DIR / "image"
LABEL_DIR = BASE_DIR / "label"
MANUAL_SPLIT_PATH = BASE_DIR / "manual_split.csv"

OUTPUT_DIR = BASE_DIR / "svm_5fold_cv_output"
DATA_LIST_DIR = OUTPUT_DIR / "data_lists"
CV_DIR = OUTPUT_DIR / "cross_validation"
TUNING_DIR = OUTPUT_DIR / "hyperparameter_tuning"
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "test_predictions"
ERROR_MAP_DIR = OUTPUT_DIR / "test_error_maps"
PLOT_DIR = OUTPUT_DIR / "plots"

for folder in [
    OUTPUT_DIR,
    DATA_LIST_DIR,
    CV_DIR,
    TUNING_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    ERROR_MAP_DIR,
    PLOT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_HEIGHT = 256
TARGET_WIDTH = 256
INPUT_CHANNELS = 3

# ------------------------------------------------------------
# SVM settings
# ------------------------------------------------------------
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = None
SVM_CACHE_SIZE_MB = 2048

# Equal tuning budget requested for fairer model comparison.
# Use the same number of candidate configurations for SVM, RF, and Hybrid.
N_CV_TRIALS = 30

# ------------------------------------------------------------
# Pixel sampling for SVM training
# ------------------------------------------------------------
# Tuning stage: smaller training sample for speed.
TUNING_ASH_PIXELS = 6000
TUNING_NONASH_PIXELS = 6000

# Best-hyperparameter 5-fold CV: larger training sample.
BEST_CV_ASH_PIXELS = 10000
BEST_CV_NONASH_PIXELS = 10000

# Final model trained on ALL Phase 1-13 images.
FINAL_ASH_PIXELS = 15000
FINAL_NONASH_PIXELS = 15000

# To prevent one image from dominating the sampled training set.
PER_IMAGE_CLASS_CAP = 1200

# During hyperparameter search, evaluate a reproducible subset of pixels
# from every validation image for speed.
# Set to None if you want ALL pixels for every tuning trial.
TUNING_EVAL_PIXELS_PER_IMAGE = 4096

# Final CV and final test use ALL pixels.
PREDICTION_CHUNK_SIZE = 200000

# Fixed phase-grouped folds.
FOLD_TO_PHASES = {
    1: [1, 13],          # 52 images
    2: [2, 12],          # 47 images
    3: [3, 7, 9],        # 46 images
    4: [4, 5, 8, 11],    # 49 images
    5: [6, 10],          # 51 images
}

PHASE_TO_FOLD = {
    phase: fold
    for fold, phases in FOLD_TO_PHASES.items()
    for phase in phases
}

# Correct total = 293 images: 245 train + 48 test.
EXPECTED_PHASE_COUNTS = {
    1: 26, 2: 37, 3: 17, 4: 14, 5: 10, 6: 9, 7: 11,
    8: 18, 9: 18, 10: 42, 11: 7, 12: 10, 13: 26,
    14: 29, 15: 6, 16: 13,
}

for required_path in [IMAGE_DIR, LABEL_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(f"Directory not found: {required_path}")

if not MANUAL_SPLIT_PATH.exists():
    raise FileNotFoundError(f"manual_split.csv not found: {MANUAL_SPLIT_PATH}")

print("=" * 90)
print("SVM (RBF) / PHASE-GROUPED 5-FOLD CV")
print("=" * 90)
print("Image directory :", IMAGE_DIR)
print("Label directory :", LABEL_DIR)
print("Output directory:", OUTPUT_DIR)
print("Target size     :", TARGET_HEIGHT, "x", TARGET_WIDTH)
print("CV trials       :", N_CV_TRIALS)
print("CV folds        :", len(FOLD_TO_PHASES))




SVM (RBF) / PHASE-GROUPED 5-FOLD CV
Image directory : D:\Experiment Ajarn\image
Label directory : D:\Experiment Ajarn\label
Output directory: D:\Experiment Ajarn\svm_5fold_cv_output
Target size     : 256 x 256
CV trials       : 30
CV folds        : 5


In [3]:
# ============================================================
# CELL 3 - TIMESTAMP AND PHASE FUNCTIONS
# ============================================================

def extract_timestamp(filename: str):
    """Extract YYYYMMDD_HHMM from an image/label filename."""
    match = re.search(r"(\d{8}_\d{4})", filename)
    return match.group(1) if match else None


def timestamp_to_phase(timestamp: str):
    """
    Assign each timestamp to eruption Phase 1-16.

    Important:
      20250618_0000 -> Phase 14
      20250802_0000 -> Phase 16
    """
    dt = pd.to_datetime(timestamp, format="%Y%m%d_%H%M")

    ranges = [
        (1,  "2024-11-08 21:30", "2024-11-09 10:00"),
        (2,  "2024-11-09 10:30", "2024-11-10 04:30"),
        (3,  "2024-11-10 05:00", "2024-11-10 13:00"),
        (4,  "2024-11-10 13:30", "2024-11-10 20:00"),
        (5,  "2024-11-11 03:00", "2024-11-11 07:30"),
        (6,  "2024-11-11 08:00", "2024-11-11 12:00"),
        (7,  "2024-11-11 12:30", "2024-11-11 17:30"),
        (8,  "2024-11-11 18:00", "2024-11-12 02:30"),
        (9,  "2024-11-12 03:00", "2024-11-12 11:30"),
        (10, "2024-11-12 12:00", "2024-11-13 08:30"),
        (11, "2024-11-13 09:00", "2024-11-13 12:00"),
        (12, "2024-11-13 14:30", "2024-11-13 19:00"),
        (13, "2024-11-13 19:30", "2024-11-14 08:00"),

        # Locked independent test data.
        (14, "2025-06-17 10:00", "2025-06-18 00:00"),
        (15, "2025-08-01 15:00", "2025-08-01 17:30"),
        (16, "2025-08-01 18:00", "2025-08-02 00:00"),
    ]

    for phase, start, end in ranges:
        if pd.Timestamp(start) <= dt <= pd.Timestamp(end):
            return phase

    return None


print("20250618_0000 -> Phase", timestamp_to_phase("20250618_0000"))
print("20250802_0000 -> Phase", timestamp_to_phase("20250802_0000"))




20250618_0000 -> Phase 14
20250802_0000 -> Phase 16


In [4]:
# ============================================================
# CELL 4 - FIND, MATCH, AND SPLIT IMAGE-LABEL PAIRS
# ============================================================

image_files = sorted(
    [
        p.resolve()
        for p in IMAGE_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    ],
    key=lambda p: p.name.lower(),
)

label_files = sorted(
    [
        p.resolve()
        for p in LABEL_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff", ".png"}
    ],
    key=lambda p: p.name.lower(),
)

if not image_files:
    raise FileNotFoundError(f"No TIFF images found in {IMAGE_DIR}")

if not label_files:
    raise FileNotFoundError(f"No label files found in {LABEL_DIR}")

image_groups = defaultdict(list)
for p in image_files:
    ts = extract_timestamp(p.name)
    if ts:
        image_groups[ts].append(p)

label_groups = defaultdict(list)
for p in label_files:
    ts = extract_timestamp(p.name)
    if ts:
        label_groups[ts].append(p)

image_dictionary = {
    ts: sorted(paths, key=lambda p: p.name.lower())[0]
    for ts, paths in image_groups.items()
}

label_dictionary = {
    ts: sorted(paths, key=lambda p: p.name.lower())[0]
    for ts, paths in label_groups.items()
}

matched_timestamps = sorted(set(image_dictionary) & set(label_dictionary))

if not matched_timestamps:
    raise ValueError("No matching image-label pairs were found.")

pairs_df = pd.DataFrame([
    {
        "timestamp": ts,
        "image_path": str(image_dictionary[ts]),
        "label_path": str(label_dictionary[ts]),
    }
    for ts in matched_timestamps
])

manual_split_df = pd.read_csv(
    MANUAL_SPLIT_PATH,
    dtype={"timestamp": str, "split": str},
)

required_columns = {"timestamp", "split"}
missing_columns = required_columns - set(manual_split_df.columns)

if missing_columns:
    raise ValueError(
        "manual_split.csv is missing columns: "
        + ", ".join(sorted(missing_columns))
    )

manual_split_df["timestamp"] = (
    manual_split_df["timestamp"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

manual_split_df["split"] = (
    manual_split_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

manual_split_df["split"] = manual_split_df["split"].map({
    "train": "train",
    "training": "train",
    "tr": "train",
    "test": "test",
    "testing": "test",
    "te": "test",
})

if manual_split_df["split"].isna().any():
    raise ValueError(
        "manual_split.csv split must contain only train/test values."
    )

if manual_split_df["timestamp"].duplicated().any():
    raise ValueError("Duplicate timestamps found in manual_split.csv.")

split_df = pairs_df.merge(
    manual_split_df[["timestamp", "split"]],
    on="timestamp",
    how="left",
    validate="one_to_one",
)

if split_df["split"].isna().any():
    bad = split_df.loc[
        split_df["split"].isna(),
        ["timestamp", "image_path"]
    ]
    print(bad)
    raise ValueError(
        "Some matched pairs have no train/test assignment."
    )

split_df["phase"] = split_df["timestamp"].apply(timestamp_to_phase)

if split_df["phase"].isna().any():
    bad = split_df.loc[
        split_df["phase"].isna(),
        ["timestamp", "split"]
    ]
    print(bad.to_string(index=False))
    raise ValueError(
        "Some images could not be assigned to Phase 1-16."
    )

split_df["phase"] = split_df["phase"].astype(int)
split_df["phase_name"] = "Phase " + split_df["phase"].astype(str)

# Strict leakage checks.
bad_train = split_df[
    (split_df["split"] == "train")
    & (~split_df["phase"].between(1, 13))
]

bad_test = split_df[
    (split_df["split"] == "test")
    & (~split_df["phase"].between(14, 16))
]

if not bad_train.empty or not bad_test.empty:
    print("Unexpected train rows:")
    print(bad_train[["timestamp", "phase", "split"]])
    print("Unexpected test rows:")
    print(bad_test[["timestamp", "phase", "split"]])

    raise ValueError(
        "Train must contain only Phases 1-13 and "
        "test only Phases 14-16."
    )

split_df["fold"] = pd.Series(
    pd.NA,
    index=split_df.index,
    dtype="Int64",
)

train_mask = split_df["split"] == "train"

split_df.loc[train_mask, "fold"] = (
    split_df.loc[train_mask, "phase"]
    .map(PHASE_TO_FOLD)
    .astype("Int64")
)

if split_df.loc[train_mask, "fold"].isna().any():
    raise ValueError(
        "At least one training phase has no fold assignment."
    )

train_df = (
    split_df[split_df["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    split_df[split_df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

split_df.to_csv(
    DATA_LIST_DIR / "all_pairs_with_phase_and_fold.csv",
    index=False,
)

train_df.to_csv(
    DATA_LIST_DIR / "training_files_with_folds.csv",
    index=False,
)

test_df.to_csv(
    DATA_LIST_DIR / "testing_files_with_phases.csv",
    index=False,
)

print("=" * 90)
print("TRAIN / TEST SPLIT")
print("=" * 90)
print("Training images:", len(train_df))
print("Testing images :", len(test_df))
print("Total images   :", len(split_df))




TRAIN / TEST SPLIT
Training images: 245
Testing images : 48
Total images   : 293


In [5]:
# ============================================================
# CELL 5 - READ ASH-RGB IMAGE AND BINARY LABEL
# ============================================================

def read_tiff_rgb(image_path) -> np.ndarray:
    """
    Read TIFF and return H x W x 3 float32 array in 0-255 scale.
    """
    if isinstance(image_path, bytes):
        image_path = image_path.decode("utf-8")

    with rasterio.open(str(image_path)) as src:
        image = src.read()

    image = np.transpose(image, (1, 2, 0))
    channel_count = image.shape[-1]

    if channel_count == 1:
        image = np.repeat(image, 3, axis=-1)

    elif channel_count == 2:
        image = np.concatenate(
            [image, np.zeros_like(image[:, :, :1])],
            axis=-1,
        )

    elif channel_count > 3:
        image = image[:, :, :3]

    image = np.nan_to_num(
        image.astype(np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    image_min = float(np.min(image))
    image_max = float(np.max(image))

    # Same normalization logic as the Hybrid notebook.
    if image_min >= 0 and image_max <= 1:
        image = image * 255.0

    elif image_min >= 0 and image_max <= 255:
        pass

    elif image_min >= 0 and image_max <= 65535:
        image = image / 65535.0 * 255.0

    elif image_max > image_min:
        image = (
            (image - image_min)
            / (image_max - image_min)
            * 255.0
        )

    else:
        image = np.zeros_like(
            image,
            dtype=np.float32,
        )

    image = np.clip(
        image,
        0.0,
        255.0,
    ).astype(np.float32)

    # Resize if necessary.
    if image.shape[:2] != (TARGET_HEIGHT, TARGET_WIDTH):
        image = np.asarray(
            Image.fromarray(
                image.astype(np.uint8)
            ).resize(
                (TARGET_WIDTH, TARGET_HEIGHT),
                resample=Image.Resampling.BILINEAR,
            ),
            dtype=np.float32,
        )

    return image


def read_binary_label(label_path) -> np.ndarray:
    """
    Return H x W uint8 binary mask:
      0 = non-ash
      1 = ash

    Supports:
      - single-band 0/1
      - single-band 0/255
      - RGB red=ash / blue=non-ash
    """
    if isinstance(label_path, bytes):
        label_path = label_path.decode("utf-8")

    label_path = Path(str(label_path))

    if label_path.suffix.lower() in {".tif", ".tiff"}:
        with rasterio.open(label_path) as src:
            raw = src.read()

        if raw.shape[0] == 1:
            label_array = raw[0]
        else:
            label_array = np.transpose(
                raw[:3],
                (1, 2, 0),
            )

    else:
        with Image.open(label_path) as img:
            label_array = np.array(img)

    if label_array.ndim == 3:
        rgb = label_array[:, :, :3].astype(np.float32)

        red = rgb[:, :, 0]
        green = rgb[:, :, 1]
        blue = rgb[:, :, 2]

        binary_label = (
            (red > green)
            & (red > blue)
            & (red >= 128)
        ).astype(np.uint8)

    elif label_array.ndim == 2:
        arr = label_array.astype(np.float32)
        unique_values = np.unique(arr)

        if np.all(np.isin(unique_values, [0, 1])):
            binary_label = arr.astype(np.uint8)
        else:
            binary_label = (
                arr >= 128
            ).astype(np.uint8)

    else:
        raise ValueError(
            f"Unsupported label shape: {label_array.shape}"
        )

    if binary_label.shape != (TARGET_HEIGHT, TARGET_WIDTH):
        binary_label = np.asarray(
            Image.fromarray(
                binary_label.astype(np.uint8)
            ).resize(
                (TARGET_WIDTH, TARGET_HEIGHT),
                resample=Image.Resampling.NEAREST,
            ),
            dtype=np.uint8,
        )

        binary_label = (
            binary_label >= 1
        ).astype(np.uint8)

    return binary_label


def load_image_and_mask(row):
    image = read_tiff_rgb(row.image_path)
    mask = read_binary_label(row.label_path)

    if image.shape[:2] != mask.shape:
        raise ValueError(
            f"Image/mask shape mismatch for {row.timestamp}: "
            f"{image.shape} vs {mask.shape}"
        )

    return image, mask




In [6]:
# ============================================================
# CELL 6 - CALCULATE PHASE/FOLD ASH PERCENTAGES
# ============================================================

def label_pixel_counts(label_path):
    mask = read_binary_label(label_path)

    ash = int(np.sum(mask == 1))
    non_ash = int(np.sum(mask == 0))
    total = ash + non_ash

    return ash, non_ash, total


pixel_stats = []

for _, row in split_df.iterrows():
    ash, non_ash, total = label_pixel_counts(
        row["label_path"]
    )
    pixel_stats.append(
        (ash, non_ash, total)
    )

split_df[
    [
        "ash_pixels",
        "non_ash_pixels",
        "total_pixels",
    ]
] = pd.DataFrame(
    pixel_stats,
    index=split_df.index,
)

split_df["ash_percent"] = (
    split_df["ash_pixels"]
    / split_df["total_pixels"].clip(lower=1)
    * 100.0
)

train_df = (
    split_df[split_df["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    split_df[split_df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

phase_summary = (
    split_df
    .groupby(
        ["phase", "phase_name", "split"],
        as_index=False,
    )
    .agg(
        number_of_images=("timestamp", "count"),
        total_pixels=("total_pixels", "sum"),
        ash_pixels=("ash_pixels", "sum"),
        non_ash_pixels=("non_ash_pixels", "sum"),
        mean_image_ash_percent=("ash_percent", "mean"),
        std_image_ash_percent=("ash_percent", "std"),
    )
)

phase_summary["phase_ash_percent"] = (
    phase_summary["ash_pixels"]
    / phase_summary["total_pixels"].clip(lower=1)
    * 100.0
)

phase_summary["phase_non_ash_percent"] = (
    100.0
    - phase_summary["phase_ash_percent"]
)

phase_summary.to_csv(
    DATA_LIST_DIR / "phase_summary.csv",
    index=False,
)

fold_rows = []

for fold, phases in FOLD_TO_PHASES.items():
    fold_df = train_df[
        train_df["fold"] == fold
    ]

    total = int(
        fold_df["total_pixels"].sum()
    )

    ash = int(
        fold_df["ash_pixels"].sum()
    )

    fold_rows.append({
        "fold": fold,
        "validation_phases": ", ".join(
            f"Phase {p}"
            for p in phases
        ),
        "validation_images": len(fold_df),
        "training_images": len(train_df) - len(fold_df),
        "validation_ash_percent":
            ash / max(total, 1) * 100,
        "validation_non_ash_percent":
            100 - ash / max(total, 1) * 100,
    })

fold_summary = pd.DataFrame(
    fold_rows
)

fold_summary.to_csv(
    DATA_LIST_DIR / "fold_summary.csv",
    index=False,
)

print("\nPHASE SUMMARY")
print(
    phase_summary[
        [
            "phase_name",
            "split",
            "number_of_images",
            "phase_ash_percent",
            "phase_non_ash_percent",
        ]
    ]
    .round(3)
    .to_string(index=False)
)

print("\nFOLD SUMMARY")
print(
    fold_summary
    .round(3)
    .to_string(index=False)
)

print("\nCOUNT CHECK")

actual_counts = (
    split_df
    .groupby("phase")
    .size()
    .to_dict()
)

for phase in range(1, 17):
    actual = actual_counts.get(
        phase,
        0,
    )

    expected = EXPECTED_PHASE_COUNTS.get(
        phase
    )

    status = (
        "OK"
        if actual == expected
        else "CHECK"
    )

    print(
        f"Phase {phase:2d}: "
        f"actual={actual:3d} "
        f"expected={expected:3d} "
        f"{status}"
    )





PHASE SUMMARY
phase_name split  number_of_images  phase_ash_percent  phase_non_ash_percent
   Phase 1 train                26              6.829                 93.171
   Phase 2 train                37              4.109                 95.891
   Phase 3 train                17              0.862                 99.138
   Phase 4 train                14              1.653                 98.347
   Phase 5 train                10              2.047                 97.953
   Phase 6 train                 9              2.539                 97.461
   Phase 7 train                11              5.323                 94.677
   Phase 8 train                18              3.561                 96.439
   Phase 9 train                18              3.149                 96.851
  Phase 10 train                42              2.206                 97.794
  Phase 11 train                 7              0.444                 99.556
  Phase 12 train                10              0.238        

In [7]:
# ============================================================
# CELL 7 - METRICS + THRESHOLD CALIBRATION
# ============================================================

def confusion_counts(actual, predicted):
    actual = (
        np.asarray(actual)
        .astype(np.uint8)
        .reshape(-1)
    )

    predicted = (
        np.asarray(predicted)
        .astype(np.uint8)
        .reshape(-1)
    )

    tp = int(np.sum((actual == 1) & (predicted == 1)))
    tn = int(np.sum((actual == 0) & (predicted == 0)))
    fp = int(np.sum((actual == 0) & (predicted == 1)))
    fn = int(np.sum((actual == 1) & (predicted == 0)))

    return tp, tn, fp, fn


def metrics_from_counts(tp, tn, fp, fn):
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / max(total, 1)
    precision = tp / max(tp + fp, 1)
    recall_pod = tp / max(tp + fn, 1)

    f1_dice = (
        2 * tp
        / max(2 * tp + fp + fn, 1)
    )

    iou = (
        tp
        / max(tp + fp + fn, 1)
    )

    far = (
        fp
        / max(tp + fp, 1)
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall_pod": recall_pod,
        "f1_dice": f1_dice,
        "iou": iou,
        "false_alarm_ratio": far,
        "true_positive": int(tp),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
    }


def optimal_threshold_from_scores(y_true, decision_scores):
    """
    Choose the SVM decision threshold using ONLY validation data.

    The threshold is selected from the validation precision-recall curve
    to maximize F1/Dice. AUPRC is also calculated from the continuous
    decision scores and is therefore threshold-independent.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.uint8,
    ).reshape(-1)

    decision_scores = np.asarray(
        decision_scores,
        dtype=np.float64,
    ).reshape(-1)

    if np.unique(y_true).size < 2:
        raise ValueError(
            "Threshold calibration requires both ash and non-ash "
            "pixels in the validation data."
        )

    precision_curve, recall_curve, thresholds = (
        precision_recall_curve(
            y_true,
            decision_scores,
        )
    )

    auprc = average_precision_score(
        y_true,
        decision_scores,
    )

    if len(thresholds) == 0:
        best_threshold = 0.0
        best_f1 = 0.0
    else:
        precision_for_thresholds = precision_curve[:-1]
        recall_for_thresholds = recall_curve[:-1]

        f1_curve = (
            2.0
            * precision_for_thresholds
            * recall_for_thresholds
            / np.maximum(
                precision_for_thresholds
                + recall_for_thresholds,
                1e-12,
            )
        )

        best_index = int(
            np.nanargmax(f1_curve)
        )

        best_threshold = float(
            thresholds[best_index]
        )

        best_f1 = float(
            f1_curve[best_index]
        )

    return {
        "threshold": best_threshold,
        "validation_auprc": float(auprc),
        "validation_best_pr_f1": best_f1,
    }


def metrics_from_scores(
    y_true,
    decision_scores,
    threshold,
):
    """
    Calculate threshold-dependent metrics plus threshold-independent AUPRC.
    """
    y_true = np.asarray(
        y_true,
        dtype=np.uint8,
    ).reshape(-1)

    decision_scores = np.asarray(
        decision_scores,
        dtype=np.float64,
    ).reshape(-1)

    predicted = (
        decision_scores >= float(threshold)
    ).astype(np.uint8)

    tp, tn, fp, fn = confusion_counts(
        y_true,
        predicted,
    )

    metrics = metrics_from_counts(
        tp,
        tn,
        fp,
        fn,
    )

    if np.unique(y_true).size >= 2:
        auprc = average_precision_score(
            y_true,
            decision_scores,
        )
    else:
        auprc = np.nan

    metrics["auprc"] = float(auprc)
    metrics["decision_threshold"] = float(threshold)

    return metrics


In [8]:
# ============================================================
# CELL 8 - PIXEL SAMPLING FOR SVM TRAINING
# ============================================================

def sample_class_pixels(
    dataframe,
    target_ash_pixels,
    target_nonash_pixels,
    seed,
):
    """
    Build a reproducible pixel sample from ONLY the provided dataframe.

    Each pixel feature:
        [R, G, B]

    This function limits the number of candidate pixels contributed by
    each image so that a small number of ash-rich images do not dominate
    the SVM training sample.
    """
    rng = np.random.default_rng(seed)

    ash_candidates = []
    nonash_candidates = []

    for row in dataframe.itertuples(index=False):
        image, mask = load_image_and_mask(row)

        X = image.reshape(
            -1,
            INPUT_CHANNELS,
        ).astype(np.float32)

        y = mask.reshape(-1)

        ash_idx = np.flatnonzero(
            y == 1
        )

        nonash_idx = np.flatnonzero(
            y == 0
        )

        if ash_idx.size > 0:
            n = min(
                PER_IMAGE_CLASS_CAP,
                ash_idx.size,
            )

            selected = rng.choice(
                ash_idx,
                size=n,
                replace=False,
            )

            ash_candidates.append(
                X[selected]
            )

        if nonash_idx.size > 0:
            n = min(
                PER_IMAGE_CLASS_CAP,
                nonash_idx.size,
            )

            selected = rng.choice(
                nonash_idx,
                size=n,
                replace=False,
            )

            nonash_candidates.append(
                X[selected]
            )

    if not ash_candidates:
        raise ValueError(
            "No ash pixels found in the training dataframe."
        )

    if not nonash_candidates:
        raise ValueError(
            "No non-ash pixels found in the training dataframe."
        )

    X_ash = np.concatenate(
        ash_candidates,
        axis=0,
    )

    X_nonash = np.concatenate(
        nonash_candidates,
        axis=0,
    )

    if len(X_ash) > target_ash_pixels:
        idx = rng.choice(
            len(X_ash),
            size=target_ash_pixels,
            replace=False,
        )
        X_ash = X_ash[idx]

    if len(X_nonash) > target_nonash_pixels:
        idx = rng.choice(
            len(X_nonash),
            size=target_nonash_pixels,
            replace=False,
        )
        X_nonash = X_nonash[idx]

    y_ash = np.ones(
        len(X_ash),
        dtype=np.uint8,
    )

    y_nonash = np.zeros(
        len(X_nonash),
        dtype=np.uint8,
    )

    X_train = np.concatenate(
        [X_ash, X_nonash],
        axis=0,
    )

    y_train = np.concatenate(
        [y_ash, y_nonash],
        axis=0,
    )

    order = rng.permutation(
        len(y_train)
    )

    X_train = X_train[order]
    y_train = y_train[order]

    sample_info = {
        "sampled_ash_pixels": int(
            np.sum(y_train == 1)
        ),
        "sampled_nonash_pixels": int(
            np.sum(y_train == 0)
        ),
        "sampled_total_pixels": int(
            len(y_train)
        ),
    }

    return X_train, y_train, sample_info




In [9]:
# ============================================================
# CELL 9 - BUILD SVM PIPELINE
# ============================================================

def build_svm_model(params):
    """
    StandardScaler and SVC are fitted only on each fold's training sample,
    preventing scaler leakage from validation/test data.

    class_weight=None because the sampled SVM training set is already
    explicitly balanced between ash and non-ash pixels.
    """
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "svm",
            SVC(
                kernel=SVM_KERNEL,
                C=float(params["C"]),
                gamma=params["gamma"],
                class_weight=SVM_CLASS_WEIGHT,
                probability=False,
                cache_size=SVM_CACHE_SIZE_MB,
            ),
        ),
    ])


In [10]:
# ============================================================
# CELL 10 - DECISION SCORE, THRESHOLD, AND EVALUATION FUNCTIONS
# ============================================================

def decision_scores_flat(
    model,
    X,
    chunk_size=PREDICTION_CHUNK_SIZE,
):
    """
    Calculate SVM decision_function scores in chunks.
    Positive scores indicate stronger support for the ash class.
    """
    X = np.asarray(
        X,
        dtype=np.float32,
    )

    scores = np.empty(
        len(X),
        dtype=np.float32,
    )

    for start in range(
        0,
        len(X),
        chunk_size,
    ):
        end = min(
            start + chunk_size,
            len(X),
        )

        scores[start:end] = (
            model.decision_function(
                X[start:end]
            )
            .astype(np.float32)
        )

    return scores


def predict_flat_pixels(
    model,
    X,
    threshold,
    chunk_size=PREDICTION_CHUNK_SIZE,
):
    """
    Convert continuous SVM decision scores to binary predictions
    using the calibrated/locked threshold.
    """
    scores = decision_scores_flat(
        model,
        X,
        chunk_size=chunk_size,
    )

    return (
        scores >= float(threshold)
    ).astype(np.uint8)


def predict_image_mask(
    model,
    image,
    threshold,
):
    """
    Predict one complete image using the supplied threshold.
    """
    h, w, c = image.shape

    X = image.reshape(
        -1,
        c,
    ).astype(np.float32)

    predicted = predict_flat_pixels(
        model,
        X,
        threshold=threshold,
    )

    return predicted.reshape(
        h,
        w,
    )


def collect_scores_on_dataframe(
    model,
    dataframe,
    max_pixels_per_image=None,
    seed=SEED,
):
    """
    Collect pooled true labels and continuous SVM decision scores.

    max_pixels_per_image:
      None -> use every pixel in every image
      int  -> reproducibly sample that many pixels per image

    This function is used for validation-only threshold calibration
    and AUPRC calculation.
    """
    rng = np.random.default_rng(
        seed
    )

    y_parts = []
    score_parts = []

    for row in dataframe.itertuples(index=False):
        image, actual = load_image_and_mask(row)

        X = image.reshape(
            -1,
            INPUT_CHANNELS,
        ).astype(np.float32)

        y = actual.reshape(-1).astype(np.uint8)

        if (
            max_pixels_per_image is not None
            and len(y) > max_pixels_per_image
        ):
            idx = rng.choice(
                len(y),
                size=max_pixels_per_image,
                replace=False,
            )

            X_eval = X[idx]
            y_eval = y[idx]

        else:
            X_eval = X
            y_eval = y

        scores = decision_scores_flat(
            model,
            X_eval,
        )

        y_parts.append(
            y_eval
        )

        score_parts.append(
            scores
        )

    y_all = np.concatenate(
        y_parts
    )

    scores_all = np.concatenate(
        score_parts
    )

    return y_all, scores_all


def calibrate_and_evaluate_on_dataframe(
    model,
    dataframe,
    max_pixels_per_image=None,
    seed=SEED,
):
    """
    Calibrate the threshold on a held-out validation fold and evaluate
    that validation fold. The independent test set is never used here.
    """
    y_true, scores = collect_scores_on_dataframe(
        model,
        dataframe,
        max_pixels_per_image=max_pixels_per_image,
        seed=seed,
    )

    calibration = optimal_threshold_from_scores(
        y_true,
        scores,
    )

    metrics = metrics_from_scores(
        y_true,
        scores,
        threshold=calibration["threshold"],
    )

    return {
        **metrics,
        "validation_best_pr_f1":
            calibration["validation_best_pr_f1"],
    }


In [11]:
# ============================================================
# CELL 11 - GENERATE REPRODUCIBLE SVM HYPERPARAMETER CANDIDATES
# ============================================================

def generate_hyperparameter_candidates(
    n_trials,
    seed=SEED,
):
    """
    Reproducible random search over a predefined RBF-SVM search space.

    The tuning budget is fixed to N_CV_TRIALS (30) so that SVM can use
    the same number of candidate configurations as the RF and Hybrid
    tuning workflows.

    Candidate 1 is retained as a reference configuration.
    """
    rng = np.random.default_rng(
        seed
    )

    C_values = [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
        1000.0,
    ]

    gamma_values = [
        "scale",
        0.0001,
        0.001,
        0.01,
        0.1,
        1.0,
    ]

    all_combinations = [
        (C, gamma)
        for C in C_values
        for gamma in gamma_values
    ]

    if n_trials > len(all_combinations):
        raise ValueError(
            f"N_CV_TRIALS={n_trials} exceeds the "
            f"{len(all_combinations)} unique combinations "
            "available in the search space."
        )

    reference = (
        10.0,
        "scale",
    )

    remaining = [
        combo
        for combo in all_combinations
        if combo != reference
    ]

    rng.shuffle(
        remaining
    )

    selected = (
        [reference]
        + remaining[: n_trials - 1]
    )

    candidates = []

    for trial_id, (C, gamma) in enumerate(
        selected,
        start=1,
    ):
        candidates.append({
            "trial_id":
                trial_id,
            "C":
                float(C),
            "gamma":
                gamma,
        })

    return pd.DataFrame(
        candidates
    )


candidate_df = generate_hyperparameter_candidates(
    N_CV_TRIALS,
    seed=SEED,
)

candidate_df.to_csv(
    TUNING_DIR / "cv_hyperparameter_candidates.csv",
    index=False,
)

print("=" * 90)
print("SVM HYPERPARAMETER CANDIDATES")
print("=" * 90)
print("Equal tuning budget:", len(candidate_df), "configurations")
print()
print(
    candidate_df.to_string(
        index=False
    )
)


SVM HYPERPARAMETER CANDIDATES
Equal tuning budget: 30 configurations

 trial_id       C   gamma
        1   10.00   scale
        2  100.00   0.001
        3    1.00     0.1
        4  100.00  0.0001
        5    0.01     1.0
        6 1000.00  0.0001
        7   10.00    0.01
        8 1000.00     1.0
        9    0.10  0.0001
       10    0.10     0.1
       11 1000.00   scale
       12   10.00   0.001
       13  100.00   scale
       14 1000.00   0.001
       15   10.00  0.0001
       16    0.10   scale
       17  100.00    0.01
       18   10.00     1.0
       19    0.10    0.01
       20  100.00     0.1
       21    0.01    0.01
       22    0.01   scale
       23    1.00    0.01
       24    1.00     1.0
       25  100.00     1.0
       26    1.00   scale
       27   10.00     0.1
       28    0.10     1.0
       29    1.00   0.001
       30 1000.00     0.1


In [12]:
# ============================================================
# CELL 12 - 5-FOLD CV HYPERPARAMETER TUNING
# VALIDATION-ONLY THRESHOLD CALIBRATION + AUPRC
# ============================================================

cv_tuning_records = []

for _, candidate in candidate_df.iterrows():
    trial_id = int(
        candidate["trial_id"]
    )

    params = {
        "C":
            float(candidate["C"]),
        "gamma":
            candidate["gamma"],
    }

    print("\n" + "=" * 100)
    print(
        f"SVM HYPERPARAMETER TRIAL "
        f"{trial_id}/{len(candidate_df)}"
    )
    print("=" * 100)
    print(params)

    for fold in sorted(
        FOLD_TO_PHASES
    ):
        fold_train_df = (
            train_df[
                train_df["fold"] != fold
            ]
            .copy()
            .reset_index(drop=True)
        )

        fold_val_df = (
            train_df[
                train_df["fold"] == fold
            ]
            .copy()
            .reset_index(drop=True)
        )

        print("\n" + "-" * 90)
        print(
            f"Trial {trial_id} | Fold {fold}"
        )
        print(
            "Train images:",
            len(fold_train_df),
        )
        print(
            "Val images  :",
            len(fold_val_df),
        )
        print(
            "Val phases  :",
            FOLD_TO_PHASES[fold],
        )

        X_train, y_train, sample_info = sample_class_pixels(
            fold_train_df,
            target_ash_pixels=TUNING_ASH_PIXELS,
            target_nonash_pixels=TUNING_NONASH_PIXELS,
            seed=(
                SEED
                + trial_id * 100
                + fold
            ),
        )

        print(
            "Training pixel sample:",
            sample_info,
        )

        model = build_svm_model(
            params
        )

        start_time = time.time()

        model.fit(
            X_train,
            y_train,
        )

        training_seconds = (
            time.time()
            - start_time
        )

        # Threshold calibration uses ONLY the held-out validation fold.
        # No test pixels are used during model selection.
        fold_metrics = calibrate_and_evaluate_on_dataframe(
            model,
            fold_val_df,
            max_pixels_per_image=TUNING_EVAL_PIXELS_PER_IMAGE,
            seed=SEED + fold,
        )

        record = {
            "trial_id":
                trial_id,
            "fold":
                fold,
            "validation_phases":
                ", ".join(
                    str(p)
                    for p in FOLD_TO_PHASES[fold]
                ),
            "training_images":
                len(fold_train_df),
            "validation_images":
                len(fold_val_df),
            "C":
                params["C"],
            "gamma":
                params["gamma"],
            "training_seconds":
                training_seconds,
            **sample_info,
            **fold_metrics,
        }

        cv_tuning_records.append(
            record
        )

        print(
            f"Threshold={fold_metrics['decision_threshold']:.5f} | "
            f"AUPRC={fold_metrics['auprc']:.4f} | "
            f"Dice={fold_metrics['f1_dice']:.4f} | "
            f"IoU={fold_metrics['iou']:.4f} | "
            f"POD={fold_metrics['recall_pod']:.4f} | "
            f"FAR={fold_metrics['false_alarm_ratio']:.4f}"
        )

        del model
        del X_train
        del y_train
        gc.collect()


cv_tuning_df = pd.DataFrame(
    cv_tuning_records
)

cv_tuning_df.to_csv(
    TUNING_DIR / "cv_tuning_fold_results.csv",
    index=False,
)


cv_trial_summary = (
    cv_tuning_df
    .groupby(
        "trial_id",
        as_index=False,
    )
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall_pod=("recall_pod", "mean"),
        std_recall_pod=("recall_pod", "std"),
        mean_f1_dice=("f1_dice", "mean"),
        std_f1_dice=("f1_dice", "std"),
        mean_iou=("iou", "mean"),
        std_iou=("iou", "std"),
        mean_false_alarm_ratio=("false_alarm_ratio", "mean"),
        std_false_alarm_ratio=("false_alarm_ratio", "std"),
        mean_auprc=("auprc", "mean"),
        std_auprc=("auprc", "std"),
        mean_calibrated_threshold=("decision_threshold", "mean"),
        median_calibrated_threshold=("decision_threshold", "median"),
        std_calibrated_threshold=("decision_threshold", "std"),
        mean_training_seconds=("training_seconds", "mean"),
    )
)


cv_trial_summary = cv_trial_summary.merge(
    candidate_df,
    on="trial_id",
    how="left",
)


# Primary model-selection criterion:
# mean validation F1/Dice after validation-only threshold calibration.
# Mean AUPRC is the threshold-independent secondary criterion.
cv_trial_summary = (
    cv_trial_summary
    .sort_values(
        [
            "mean_f1_dice",
            "mean_auprc",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


cv_trial_summary.to_csv(
    TUNING_DIR / "cv_tuning_trial_summary.csv",
    index=False,
)


print("\n" + "=" * 100)
print("SVM HYPERPARAMETER TUNING SUMMARY")
print("=" * 100)

print(
    cv_trial_summary[
        [
            "trial_id",
            "C",
            "gamma",
            "mean_f1_dice",
            "std_f1_dice",
            "mean_auprc",
            "std_auprc",
            "mean_calibrated_threshold",
            "mean_iou",
            "mean_recall_pod",
            "mean_false_alarm_ratio",
        ]
    ]
    .round(5)
    .to_string(index=False)
)



SVM HYPERPARAMETER TRIAL 1/30
{'C': 10.0, 'gamma': 'scale'}

------------------------------------------------------------------------------------------
Trial 1 | Fold 1
Train images: 193
Val images  : 52
Val phases  : [1, 13]
Training pixel sample: {'sampled_ash_pixels': 6000, 'sampled_nonash_pixels': 6000, 'sampled_total_pixels': 12000}


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=0.41933 | AUPRC=0.5059 | Dice=0.5187 | IoU=0.3501 | POD=0.4460 | FAR=0.3803

------------------------------------------------------------------------------------------
Trial 1 | Fold 2
Train images: 198
Val images  : 47
Val phases  : [2, 12]
Training pixel sample: {'sampled_ash_pixels': 6000, 'sampled_nonash_pixels': 6000, 'sampled_total_pixels': 12000}


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=0.97872 | AUPRC=0.4073 | Dice=0.4563 | IoU=0.2956 | POD=0.4633 | FAR=0.5505

------------------------------------------------------------------------------------------
Trial 1 | Fold 3
Train images: 199
Val images  : 46
Val phases  : [3, 7, 9]
Training pixel sample: {'sampled_ash_pixels': 6000, 'sampled_nonash_pixels': 6000, 'sampled_total_pixels': 12000}


...

Threshold=1.21810 | AUPRC=0.6213 | Dice=0.5858 | IoU=0.4142 | POD=0.7620 | FAR=0.5243

------------------------------------------------------------------------------------------
Trial 30 | Fold 4
Train images: 196
Val images  : 49
Val phases  : [4, 5, 8, 11]
Training pixel sample: {'sampled_ash_pixels': 6000, 'sampled_nonash_pixels': 6000, 'sampled_total_pixels': 12000}


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=1.06656 | AUPRC=0.4814 | Dice=0.4686 | IoU=0.3060 | POD=0.7928 | FAR=0.6674

------------------------------------------------------------------------------------------
Trial 30 | Fold 5
Train images: 194
Val images  : 51
Val phases  : [6, 10]
Training pixel sample: {'sampled_ash_pixels': 6000, 'sampled_nonash_pixels': 6000, 'sampled_total_pixels': 12000}


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=1.29554 | AUPRC=0.5454 | Dice=0.5358 | IoU=0.3659 | POD=0.7319 | FAR=0.5774

SVM HYPERPARAMETER TUNING SUMMARY
 trial_id       C   gamma  mean_f1_dice  std_f1_dice  mean_auprc  std_auprc  mean_calibrated_threshold  mean_iou  mean_recall_pod  mean_false_alarm_ratio
        3    1.00     0.1       0.56653      0.09628     0.60310    0.10957                    1.45222   0.40027          0.56361                 0.41058
       10    0.10     0.1       0.56244      0.09295     0.60318    0.11419                    1.36179   0.39588          0.52647                 0.39193
       17  100.00    0.01       0.55118      0.07980     0.57854    0.10895                    1.77174   0.38381          0.52471                 0.40131
       27   10.00     0.1       0.54856      0.09207     0.55095    0.09549                    1.49178   0.38232          0.53438                 0.42535
       22    0.01   scale       0.53567      0.08076     0.53732    0.07689                    0.86337   0.36

In [13]:
# ============================================================
# CELL 13 - SELECT AND SAVE BEST SVM HYPERPARAMETERS
# ============================================================

best_trial_id = int(
    cv_trial_summary.iloc[0][
        "trial_id"
    ]
)

best_row = candidate_df[
    candidate_df["trial_id"]
    == best_trial_id
].iloc[0]

best_params = {
    "C":
        float(best_row["C"]),
    "gamma":
        best_row["gamma"],
}

best_tuning_summary_row = (
    cv_trial_summary.iloc[0]
)

with open(
    TUNING_DIR / "best_hyperparameters.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "best_trial_id":
                best_trial_id,
            "kernel":
                SVM_KERNEL,
            "class_weight":
                SVM_CLASS_WEIGHT,
            "selection_metric":
                "mean_f1_dice_after_validation_threshold_calibration",
            "mean_cv_f1_dice":
                float(best_tuning_summary_row["mean_f1_dice"]),
            "mean_cv_auprc":
                float(best_tuning_summary_row["mean_auprc"]),
            **best_params,
        },
        f,
        indent=4,
    )

pd.DataFrame([
    {
        "best_trial_id":
            best_trial_id,
        "kernel":
            SVM_KERNEL,
        "class_weight":
            SVM_CLASS_WEIGHT,
        "selection_metric":
            "mean_f1_dice_after_validation_threshold_calibration",
        "mean_cv_f1_dice":
            best_tuning_summary_row["mean_f1_dice"],
        "mean_cv_auprc":
            best_tuning_summary_row["mean_auprc"],
        **best_params,
    }
]).to_csv(
    TUNING_DIR / "best_hyperparameters.csv",
    index=False,
)

print("=" * 90)
print("BEST CROSS-VALIDATED SVM HYPERPARAMETERS")
print("=" * 90)
print("Best trial  :", best_trial_id)
print("Kernel      :", SVM_KERNEL)
print("C           :", best_params["C"])
print("Gamma       :", best_params["gamma"])
print("Class weight:", SVM_CLASS_WEIGHT)
print(
    "Mean CV F1  :",
    f"{best_tuning_summary_row['mean_f1_dice']:.5f}",
)
print(
    "Mean CV AUPRC:",
    f"{best_tuning_summary_row['mean_auprc']:.5f}",
)


BEST CROSS-VALIDATED SVM HYPERPARAMETERS
Best trial  : 3
Kernel      : rbf
C           : 1.0
Gamma       : 0.1
Class weight: None
Mean CV F1  : 0.56653
Mean CV AUPRC: 0.60310


In [14]:
# ============================================================
# CELL 14 - FULL 5-FOLD CV USING BEST SVM HYPERPARAMETERS
# ALL VALIDATION PIXELS + FINAL THRESHOLD CALIBRATION
# ============================================================
#
# Each fold:
#   1) fit on that fold's training images,
#   2) calculate decision scores on ALL validation pixels,
#   3) select the validation-optimal threshold,
#   4) calculate fold metrics + AUPRC.
#
# The final threshold is the MEDIAN of the five validation-optimal
# thresholds. Test data are not used.
# ============================================================

best_hp_cv_records = []

for fold in sorted(
    FOLD_TO_PHASES
):
    fold_train_df = (
        train_df[
            train_df["fold"] != fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    fold_val_df = (
        train_df[
            train_df["fold"] == fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    print("\n" + "=" * 100)
    print(
        f"BEST-HYPERPARAMETER SVM CV | FOLD {fold}"
    )
    print("=" * 100)
    print(
        "Validation phases:",
        FOLD_TO_PHASES[fold],
    )

    X_train, y_train, sample_info = sample_class_pixels(
        fold_train_df,
        target_ash_pixels=BEST_CV_ASH_PIXELS,
        target_nonash_pixels=BEST_CV_NONASH_PIXELS,
        seed=SEED + 1000 + fold,
    )

    model = build_svm_model(
        best_params
    )

    start_time = time.time()

    model.fit(
        X_train,
        y_train,
    )

    training_seconds = (
        time.time()
        - start_time
    )

    # ALL validation pixels; threshold calibrated on validation only.
    fold_metrics = calibrate_and_evaluate_on_dataframe(
        model,
        fold_val_df,
        max_pixels_per_image=None,
        seed=SEED,
    )

    best_hp_cv_records.append({
        "fold":
            fold,
        "validation_phases":
            ", ".join(
                str(p)
                for p in FOLD_TO_PHASES[fold]
            ),
        "training_images":
            len(fold_train_df),
        "validation_images":
            len(fold_val_df),
        "training_seconds":
            training_seconds,
        **sample_info,
        **fold_metrics,
    })

    print(
        f"Threshold={fold_metrics['decision_threshold']:.5f} | "
        f"AUPRC={fold_metrics['auprc']:.4f} | "
        f"Accuracy={fold_metrics['accuracy']:.4f} | "
        f"Precision={fold_metrics['precision']:.4f} | "
        f"POD={fold_metrics['recall_pod']:.4f} | "
        f"Dice={fold_metrics['f1_dice']:.4f} | "
        f"IoU={fold_metrics['iou']:.4f} | "
        f"FAR={fold_metrics['false_alarm_ratio']:.4f}"
    )

    del model
    del X_train
    del y_train
    gc.collect()


best_hp_cv_df = pd.DataFrame(
    best_hp_cv_records
)

best_hp_cv_df.to_csv(
    CV_DIR / "best_hyperparameters_5fold_results.csv",
    index=False,
)


# ------------------------------------------------------------
# LOCK FINAL THRESHOLD BEFORE TESTING
# ------------------------------------------------------------

FINAL_DECISION_THRESHOLD = float(
    best_hp_cv_df[
        "decision_threshold"
    ].median()
)

threshold_summary = pd.DataFrame([
    {
        "threshold_aggregation":
            "median_of_5_validation_optimal_thresholds",
        "final_decision_threshold":
            FINAL_DECISION_THRESHOLD,
        "mean_threshold":
            best_hp_cv_df["decision_threshold"].mean(),
        "std_threshold":
            best_hp_cv_df["decision_threshold"].std(ddof=1),
        "min_threshold":
            best_hp_cv_df["decision_threshold"].min(),
        "max_threshold":
            best_hp_cv_df["decision_threshold"].max(),
    }
])

threshold_summary.to_csv(
    CV_DIR / "final_threshold_from_5fold_cv.csv",
    index=False,
)

with open(
    CV_DIR / "final_threshold_from_5fold_cv.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "method":
                "median_of_5_validation_optimal_thresholds",
            "final_decision_threshold":
                FINAL_DECISION_THRESHOLD,
            "fold_thresholds":
                [
                    float(value)
                    for value in best_hp_cv_df[
                        "decision_threshold"
                    ].tolist()
                ],
        },
        f,
        indent=4,
    )


# ------------------------------------------------------------
# CV SUMMARY
# ------------------------------------------------------------

metrics_for_summary = [
    "accuracy",
    "precision",
    "recall_pod",
    "f1_dice",
    "iou",
    "false_alarm_ratio",
    "auprc",
    "decision_threshold",
]

summary_rows = []

for metric in metrics_for_summary:
    summary_rows.append({
        "metric":
            metric,
        "mean":
            best_hp_cv_df[metric].mean(),
        "std":
            best_hp_cv_df[metric].std(ddof=1),
        "min":
            best_hp_cv_df[metric].min(),
        "max":
            best_hp_cv_df[metric].max(),
    })

best_hp_cv_summary = pd.DataFrame(
    summary_rows
)

best_hp_cv_summary.to_csv(
    CV_DIR / "best_hyperparameters_5fold_summary.csv",
    index=False,
)

print("\n" + "=" * 100)
print("FINAL SVM 5-FOLD CV RESULTS")
print("=" * 100)

print(
    best_hp_cv_df[
        [
            "fold",
            "validation_phases",
            "validation_images",
            "decision_threshold",
            "auprc",
            "accuracy",
            "precision",
            "recall_pod",
            "f1_dice",
            "iou",
            "false_alarm_ratio",
        ]
    ]
    .round(5)
    .to_string(index=False)
)

print("\nMEAN ± SD")
for _, row in best_hp_cv_summary.iterrows():
    print(
        f"{row['metric']:20s}: "
        f"{row['mean']:.5f} ± {row['std']:.5f}"
    )

print("\nLOCKED FINAL DECISION THRESHOLD")
print(
    f"{FINAL_DECISION_THRESHOLD:.6f}"
)
print(
    "(Median of the five validation-optimal thresholds; "
    "test data not used.)"
)



BEST-HYPERPARAMETER SVM CV | FOLD 1
Validation phases: [1, 13]


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=0.57391 | AUPRC=0.5261 | Accuracy=0.9695 | Precision=0.6546 | POD=0.3891 | Dice=0.4881 | IoU=0.3228 | FAR=0.3454

BEST-HYPERPARAMETER SVM CV | FOLD 2
Validation phases: [2, 12]


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=1.08033 | AUPRC=0.4527 | Accuracy=0.9668 | Precision=0.4937 | POD=0.3918 | Dice=0.4369 | IoU=0.2795 | FAR=0.5063

BEST-HYPERPARAMETER SVM CV | FOLD 3
Validation phases: [3, 7, 9]


...

Threshold=1.82022 | AUPRC=0.7433 | Accuracy=0.9818 | Precision=0.6717 | POD=0.6975 | Dice=0.6844 | IoU=0.5202 | FAR=0.3283

BEST-HYPERPARAMETER SVM CV | FOLD 4
Validation phases: [4, 5, 8, 11]


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=1.79934 | AUPRC=0.6066 | Accuracy=0.9822 | Precision=0.6019 | POD=0.6274 | Dice=0.6144 | IoU=0.4434 | FAR=0.3981

BEST-HYPERPARAMETER SVM CV | FOLD 5
Validation phases: [6, 10]


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Threshold=1.82840 | AUPRC=0.4292 | Accuracy=0.9746 | Precision=0.4601 | POD=0.7082 | Dice=0.5578 | IoU=0.3868 | FAR=0.5399

FINAL SVM 5-FOLD CV RESULTS
 fold validation_phases  validation_images  decision_threshold   auprc  accuracy  precision  recall_pod  f1_dice     iou  false_alarm_ratio
    1             1, 13                 52             0.57391 0.52610   0.96951    0.65456     0.38914  0.48810 0.32284            0.34544
    2             2, 12                 47             1.08033 0.45265   0.96682    0.49369     0.39175  0.43685 0.27947            0.50631
    3           3, 7, 9                 46             1.82022 0.74330   0.98183    0.67167     0.69753  0.68436 0.52017            0.32833
    4       4, 5, 8, 11                 49             1.79934 0.60663   0.98219    0.60195     0.62738  0.61440 0.44342            0.39805
    5             6, 10                 51             1.82840 0.42916   0.97457    0.46014     0.70815  0.55782 0.38679            0.53986

MEAN ± 

In [15]:
# ============================================================
# CELL 15 - FINAL SVM TRAINING ON ALL PHASE 1-13 IMAGES
# ============================================================
#
# Hyperparameters and FINAL_DECISION_THRESHOLD have already been
# chosen exclusively from phase-grouped CV.
# Phases 14-16 remain completely locked.
# ============================================================

print("\n" + "=" * 100)
print("FINAL SVM TRAINING ON ALL PHASE 1-13 IMAGES")
print("=" * 100)

X_final, y_final, final_sample_info = sample_class_pixels(
    train_df,
    target_ash_pixels=FINAL_ASH_PIXELS,
    target_nonash_pixels=FINAL_NONASH_PIXELS,
    seed=SEED + 5000,
)

print(
    "Final training pixel sample:",
    final_sample_info,
)

final_model = build_svm_model(
    best_params
)

start_time = time.time()

final_model.fit(
    X_final,
    y_final,
)

final_training_seconds = (
    time.time()
    - start_time
)

FINAL_MODEL_PATH = (
    MODEL_DIR
    / "best_cv_tuned_svm_rbf.joblib"
)

joblib.dump(
    final_model,
    FINAL_MODEL_PATH,
)

pd.DataFrame([
    {
        "training_images":
            len(train_df),
        "training_seconds":
            final_training_seconds,
        **final_sample_info,
        "C":
            best_params["C"],
        "gamma":
            best_params["gamma"],
        "kernel":
            SVM_KERNEL,
        "class_weight":
            SVM_CLASS_WEIGHT,
        "final_decision_threshold":
            FINAL_DECISION_THRESHOLD,
        "threshold_source":
            "median_of_5_validation_optimal_thresholds",
    }
]).to_csv(
    MODEL_DIR / "final_training_summary.csv",
    index=False,
)

print("Final model saved to:")
print(FINAL_MODEL_PATH)
print(
    "Locked decision threshold:",
    f"{FINAL_DECISION_THRESHOLD:.6f}",
)

del X_final
del y_final
gc.collect()



FINAL SVM TRAINING ON ALL PHASE 1-13 IMAGES
Final training pixel sample: {'sampled_ash_pixels': 15000, 'sampled_nonash_pixels': 15000, 'sampled_total_pixels': 30000}


C:\Users\computer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Final model saved to:
D:\Experiment Ajarn\svm_5fold_cv_output\models\best_cv_tuned_svm_rbf.joblib
Locked decision threshold: 1.799341


45

In [16]:
# ============================================================
# CELL 16 - FINAL TEST PREDICTIONS + PER-IMAGE METRICS + AUPRC
# ============================================================

def create_red_blue_mask(binary_mask):
    """
    Blue = non-ash
    Red  = ash
    """
    color = np.zeros(
        (*binary_mask.shape, 3),
        dtype=np.uint8,
    )

    color[binary_mask == 0] = [0, 0, 255]
    color[binary_mask == 1] = [255, 0, 0]

    return color


def create_error_map(actual, predicted):
    """
    Error-map classes:
      TN = black
      TP = green
      FP = red
      FN = yellow
    """
    error_map = np.zeros(
        (*actual.shape, 3),
        dtype=np.uint8,
    )

    tn = (actual == 0) & (predicted == 0)
    tp = (actual == 1) & (predicted == 1)
    fp = (actual == 0) & (predicted == 1)
    fn = (actual == 1) & (predicted == 0)

    error_map[tn] = [0, 0, 0]
    error_map[tp] = [0, 180, 0]
    error_map[fp] = [255, 0, 0]
    error_map[fn] = [255, 255, 0]

    return error_map


prediction_records = []

# Continuous scores are kept in memory so pooled AUPRC can be calculated.
test_labels_all = []
test_scores_all = []

test_labels_by_phase = defaultdict(list)
test_scores_by_phase = defaultdict(list)


print("\n" + "=" * 100)
print("FINAL INDEPENDENT SVM TEST: PHASES 14-16")
print("=" * 100)
print("Number of test images:", len(test_df))
print(
    "LOCKED threshold:",
    f"{FINAL_DECISION_THRESHOLD:.6f}",
)

for image_index, row in enumerate(
    test_df.itertuples(index=False),
    start=1,
):
    image, actual = load_image_and_mask(
        row
    )

    h, w, c = image.shape

    X_test_image = image.reshape(
        -1,
        c,
    ).astype(np.float32)

    y_test_image = actual.reshape(
        -1
    ).astype(np.uint8)

    decision_scores = decision_scores_flat(
        final_model,
        X_test_image,
    )

    predicted_flat = (
        decision_scores
        >= FINAL_DECISION_THRESHOLD
    ).astype(np.uint8)

    predicted = predicted_flat.reshape(
        h,
        w,
    )

    tp, tn, fp, fn = confusion_counts(
        actual,
        predicted,
    )

    metrics = metrics_from_counts(
        tp,
        tn,
        fp,
        fn,
    )

    if np.unique(y_test_image).size >= 2:
        image_auprc = average_precision_score(
            y_test_image,
            decision_scores,
        )
    else:
        image_auprc = np.nan

    metrics["auprc"] = float(
        image_auprc
    )

    metrics["decision_threshold"] = float(
        FINAL_DECISION_THRESHOLD
    )

    test_labels_all.append(
        y_test_image
    )

    test_scores_all.append(
        decision_scores
    )

    test_labels_by_phase[
        int(row.phase)
    ].append(
        y_test_image
    )

    test_scores_by_phase[
        int(row.phase)
    ].append(
        decision_scores
    )

    total_pixels = max(
        tp + tn + fp + fn,
        1,
    )

    ground_truth_ash_percent = (
        (tp + fn)
        / total_pixels
        * 100
    )

    predicted_ash_percent = (
        (tp + fp)
        / total_pixels
        * 100
    )

    prediction_records.append({
        "timestamp":
            row.timestamp,
        "phase":
            int(row.phase),
        "phase_name":
            row.phase_name,
        "ground_truth_ash_percent":
            ground_truth_ash_percent,
        "predicted_ash_percent":
            predicted_ash_percent,
        **metrics,
    })

    print(
        f"[{image_index:02d}/{len(test_df)}] "
        f"{row.timestamp} | "
        f"Phase {int(row.phase)} | "
        f"AUPRC={metrics['auprc']:.4f} | "
        f"IoU={metrics['iou']:.4f} | "
        f"Dice={metrics['f1_dice']:.4f} | "
        f"POD={metrics['recall_pod']:.4f} | "
        f"FAR={metrics['false_alarm_ratio']:.4f}"
    )

    Image.fromarray(
        predicted * 255
    ).save(
        PREDICTION_DIR
        / (
            f"{row.timestamp}_"
            f"phase{int(row.phase)}_"
            f"svm_prediction_binary.png"
        )
    )

    prediction_color = create_red_blue_mask(
        predicted
    )

    actual_color = create_red_blue_mask(
        actual
    )

    Image.fromarray(
        prediction_color
    ).save(
        PREDICTION_DIR
        / (
            f"{row.timestamp}_"
            f"phase{int(row.phase)}_"
            f"svm_prediction_color.png"
        )
    )

    error_map = create_error_map(
        actual,
        predicted,
    )

    Image.fromarray(
        error_map
    ).save(
        ERROR_MAP_DIR
        / (
            f"{row.timestamp}_"
            f"phase{int(row.phase)}_"
            f"svm_error_map.png"
        )
    )

    raw_image = np.clip(
        image,
        0,
        255,
    ).astype(np.uint8)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(20, 5),
    )

    axes[0].imshow(
        raw_image
    )

    axes[0].set_title(
        f"Himawari-9 Ash RGB\n"
        f"{row.timestamp} | "
        f"Phase {int(row.phase)}"
    )

    axes[0].axis(
        "off"
    )

    axes[1].imshow(
        actual_color
    )

    axes[1].set_title(
        "Ground Truth\n"
        "Red = Ash, Blue = Non-ash"
    )

    axes[1].axis(
        "off"
    )

    axes[2].imshow(
        prediction_color
    )

    axes[2].set_title(
        "SVM (RBF) Prediction\n"
        f"IoU={metrics['iou']:.3f}, "
        f"Dice={metrics['f1_dice']:.3f}"
    )

    axes[2].axis(
        "off"
    )

    axes[3].imshow(
        error_map
    )

    axes[3].set_title(
        "Error Map"
    )

    axes[3].axis(
        "off"
    )

    axes[3].legend(
        handles=[
            Patch(facecolor="black", label="TN"),
            Patch(facecolor="green", label="TP"),
            Patch(facecolor="red", label="FP"),
            Patch(facecolor="yellow", label="FN"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=4,
        frameon=False,
    )

    plt.tight_layout()

    plt.savefig(
        ERROR_MAP_DIR
        / (
            f"{row.timestamp}_"
            f"phase{int(row.phase)}_"
            f"svm_comparison_error_map.png"
        ),
        dpi=250,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )


test_per_image_df = pd.DataFrame(
    prediction_records
)

test_per_image_df.to_csv(
    OUTPUT_DIR / "test_per_image_metrics.csv",
    index=False,
)

TEST_Y_ALL = np.concatenate(
    test_labels_all
)

TEST_SCORES_ALL = np.concatenate(
    test_scores_all
)

print("\n" + "=" * 100)
print("SVM TEST PER-IMAGE RESULTS")
print("=" * 100)

print(
    test_per_image_df
    .round(5)
    .to_string(index=False)
)



FINAL INDEPENDENT SVM TEST: PHASES 14-16
Number of test images: 48
LOCKED threshold: 1.799341
[01/48] 20250617_1000 | Phase 14 | AUPRC=0.0144 | IoU=0.0100 | Dice=0.0199 | POD=0.0625 | FAR=0.9882
[02/48] 20250617_1030 | Phase 14 | AUPRC=0.0692 | IoU=0.0391 | Dice=0.0752 | POD=0.1081 | FAR=0.9424
[03/48] 20250617_1100 | Phase 14 | AUPRC=0.1239 | IoU=0.0651 | Dice=0.1222 | POD=0.1301 | FAR=0.8848
[04/48] 20250617_1130 | Phase 14 | AUPRC=0.2723 | IoU=0.1248 | Dice=0.2219 | POD=0.2148 | FAR=0.7704
[05/48] 20250617_1200 | Phase 14 | AUPRC=0.4142 | IoU=0.2080 | Dice=0.3444 | POD=0.3104 | FAR=0.6134
[06/48] 20250617_1230 | Phase 14 | AUPRC=0.4908 | IoU=0.2740 | Dice=0.4302 | POD=0.3728 | FAR=0.4915
[07/48] 20250617_1300 | Phase 14 | AUPRC=0.5531 | IoU=0.3407 | Dice=0.5082 | POD=0.4649 | FAR=0.4397
[08/48] 20250617_1330 | Phase 14 | AUPRC=0.6360 | IoU=0.4077 | Dice=0.5792 | POD=0.5189 | FAR=0.3446
[09/48] 20250617_1400 | Phase 14 | AUPRC=0.7248 | IoU=0.5260 | Dice=0.6894 | POD=0.6578 | FAR=0.2

In [17]:
# ============================================================
# CELL 17 - TEST METRICS BY PHASE 14, 15, AND 16 + AUPRC
# ============================================================

phase_test_records = []

for phase in [
    14,
    15,
    16,
]:
    phase_df = (
        test_per_image_df[
            test_per_image_df["phase"] == phase
        ]
        .copy()
    )

    if phase_df.empty:
        print(
            f"WARNING: No test images found "
            f"for Phase {phase}"
        )
        continue

    tp = int(
        phase_df[
            "true_positive"
        ].sum()
    )

    tn = int(
        phase_df[
            "true_negative"
        ].sum()
    )

    fp = int(
        phase_df[
            "false_positive"
        ].sum()
    )

    fn = int(
        phase_df[
            "false_negative"
        ].sum()
    )

    pooled = metrics_from_counts(
        tp,
        tn,
        fp,
        fn,
    )

    phase_y = np.concatenate(
        test_labels_by_phase[phase]
    )

    phase_scores = np.concatenate(
        test_scores_by_phase[phase]
    )

    if np.unique(phase_y).size >= 2:
        phase_auprc = average_precision_score(
            phase_y,
            phase_scores,
        )
    else:
        phase_auprc = np.nan

    total = max(
        tp + tn + fp + fn,
        1,
    )

    phase_test_records.append({
        "phase":
            phase,
        "phase_name":
            f"Phase {phase}",
        "number_of_images":
            len(phase_df),
        "decision_threshold":
            FINAL_DECISION_THRESHOLD,
        "ground_truth_ash_percent":
            (tp + fn) / total * 100,
        "predicted_ash_percent":
            (tp + fp) / total * 100,
        "pooled_auprc":
            phase_auprc,
        "mean_image_auprc":
            phase_df["auprc"].mean(),
        "std_image_auprc":
            phase_df["auprc"].std(ddof=1),
        "mean_image_dice":
            phase_df["f1_dice"].mean(),
        "std_image_dice":
            phase_df["f1_dice"].std(ddof=1),
        "mean_image_iou":
            phase_df["iou"].mean(),
        "std_image_iou":
            phase_df["iou"].std(ddof=1),
        **pooled,
    })


phase_test_df = pd.DataFrame(
    phase_test_records
)

phase_test_df.to_csv(
    OUTPUT_DIR / "test_metrics_by_phase_14_15_16.csv",
    index=False,
)

print("\n" + "=" * 100)
print("SVM TEST RESULTS BY PHASE")
print("=" * 100)

print(
    phase_test_df
    .round(5)
    .to_string(index=False)
)



SVM TEST RESULTS BY PHASE
 phase phase_name  number_of_images  decision_threshold  ground_truth_ash_percent  predicted_ash_percent  pooled_auprc  mean_image_auprc  std_image_auprc  mean_image_dice  std_image_dice  mean_image_iou  std_image_iou  accuracy  precision  recall_pod  f1_dice     iou  false_alarm_ratio  true_positive  true_negative  false_positive  false_negative
    14   Phase 14                29             1.79934                  11.81093               10.35898       0.65090           0.59295          0.21802          0.56374         0.21544         0.41879        0.18309   0.92026    0.68522     0.60098  0.64034 0.47096            0.31478         134904        1614099           61973           89568
    15   Phase 15                 6             1.79934                   2.93681                0.31281       0.52923           0.55524          0.26769          0.20998         0.23666         0.13592        0.16956   0.97373    0.99593     0.10608  0.19174 0.10603        

In [18]:
# ============================================================
# CELL 18 - OVERALL INDEPENDENT TEST METRICS + POOLED AUPRC
# ============================================================

tp = int(
    test_per_image_df[
        "true_positive"
    ].sum()
)

tn = int(
    test_per_image_df[
        "true_negative"
    ].sum()
)

fp = int(
    test_per_image_df[
        "false_positive"
    ].sum()
)

fn = int(
    test_per_image_df[
        "false_negative"
    ].sum()
)

overall_test_metrics = metrics_from_counts(
    tp,
    tn,
    fp,
    fn,
)

overall_test_auprc = (
    average_precision_score(
        TEST_Y_ALL,
        TEST_SCORES_ALL,
    )
    if np.unique(TEST_Y_ALL).size >= 2
    else np.nan
)

overall_test_metrics[
    "auprc"
] = float(
    overall_test_auprc
)

overall_test_metrics[
    "decision_threshold"
] = float(
    FINAL_DECISION_THRESHOLD
)

total = max(
    tp + tn + fp + fn,
    1,
)

overall_test_df = pd.DataFrame([
    {
        "number_of_images":
            len(test_per_image_df),
        "phases":
            "14,15,16",
        "ground_truth_ash_percent":
            (tp + fn) / total * 100,
        "predicted_ash_percent":
            (tp + fp) / total * 100,
        **overall_test_metrics,
    }
])

overall_test_df.to_csv(
    OUTPUT_DIR / "test_metrics_overall.csv",
    index=False,
)

print("\n" + "=" * 100)
print("OVERALL LOCKED SVM TEST RESULT")
print("=" * 100)

print(
    overall_test_df
    .round(5)
    .to_string(index=False)
)



OVERALL LOCKED SVM TEST RESULT
 number_of_images   phases  ground_truth_ash_percent  predicted_ash_percent  accuracy  precision  recall_pod  f1_dice     iou  false_alarm_ratio  true_positive  true_negative  false_positive  false_negative   auprc  decision_threshold
               48 14,15,16                   8.67392                6.74098   0.94099    0.70568     0.54842  0.61719 0.44633            0.29432         149641        2810458           62412          123217 0.63943             1.79934


In [19]:
# ============================================================
# CELL 19 - SAVE COMPACT CV VS TEST SUMMARY
# ============================================================

cv_summary_wide = {}

for _, row in best_hp_cv_summary.iterrows():
    cv_summary_wide[
        f"cv_{row['metric']}_mean"
    ] = row["mean"]

    cv_summary_wide[
        f"cv_{row['metric']}_std"
    ] = row["std"]

final_summary = pd.DataFrame([
    {
        "model":
            "SVM_RBF",
        "best_trial_id":
            best_trial_id,
        "best_C":
            best_params["C"],
        "best_gamma":
            best_params["gamma"],
        "kernel":
            SVM_KERNEL,
        "class_weight":
            SVM_CLASS_WEIGHT,
        "tuning_budget":
            N_CV_TRIALS,
        "threshold_method":
            "median_of_5_validation_optimal_thresholds",
        "final_decision_threshold":
            FINAL_DECISION_THRESHOLD,
        **cv_summary_wide,
        **{
            f"test_{k}": v
            for k, v in overall_test_metrics.items()
            if k not in {
                "true_positive",
                "true_negative",
                "false_positive",
                "false_negative",
            }
        },
    }
])

final_summary.to_csv(
    OUTPUT_DIR / "cv_and_independent_test_summary.csv",
    index=False,
)

print("\nAll outputs saved under:")
print(OUTPUT_DIR)

print("\nImportant files:")
print(" - hyperparameter_tuning/cv_hyperparameter_candidates.csv")
print(" - hyperparameter_tuning/cv_tuning_fold_results.csv")
print(" - hyperparameter_tuning/cv_tuning_trial_summary.csv")
print(" - hyperparameter_tuning/best_hyperparameters.csv")
print(" - cross_validation/best_hyperparameters_5fold_results.csv")
print(" - cross_validation/best_hyperparameters_5fold_summary.csv")
print(" - cross_validation/final_threshold_from_5fold_cv.csv")
print(" - models/best_cv_tuned_svm_rbf.joblib")
print(" - models/final_training_summary.csv")
print(" - test_per_image_metrics.csv")
print(" - test_metrics_by_phase_14_15_16.csv")
print(" - test_metrics_overall.csv")
print(" - cv_and_independent_test_summary.csv")
print(" - test_error_maps/*.png")



All outputs saved under:
D:\Experiment Ajarn\svm_5fold_cv_output

Important files:
 - hyperparameter_tuning/cv_hyperparameter_candidates.csv
 - hyperparameter_tuning/cv_tuning_fold_results.csv
 - hyperparameter_tuning/cv_tuning_trial_summary.csv
 - hyperparameter_tuning/best_hyperparameters.csv
 - cross_validation/best_hyperparameters_5fold_results.csv
 - cross_validation/best_hyperparameters_5fold_summary.csv
 - cross_validation/final_threshold_from_5fold_cv.csv
 - models/best_cv_tuned_svm_rbf.joblib
 - models/final_training_summary.csv
 - test_per_image_metrics.csv
 - test_metrics_by_phase_14_15_16.csv
 - test_metrics_overall.csv
 - cv_and_independent_test_summary.csv
 - test_error_maps/*.png
